# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SajidurCodes/flyrank-ml-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\Projects AI ML\flyrank-ml-starter")
load_dotenv(PROJECT_ROOT / ".env", override=True)
HF_TOKEN = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute("CREATE SECRET hf_token (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APRIL = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet"  
DIM_CONTENT = f"{WAREHOUSE}/dim_content.parquet"

DECISION_CUTOFF = "2026-03-31"  

print(con.sql(f"SELECT COUNT(*) FROM read_parquet('{FACT_APRIL}')").df())


   count_star()
0      10424730


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression**, on a binary label (`declined` vs. not), with client-grouped validation.

**Why this fits Refresh/Content Opportunity Scoring:**
- The lane's decision is fundamentally binary at the page level — does this page need review or not — which logistic regression handles directly, and its output is a probability, which slots straight into a ranked queue (same shape as the w04 baseline's score column).
- **Interpretability matters more than raw accuracy here** — reviewers need to trust *why* a page is flagged, not just that it is. Logistic regression coefficients are directly readable; that's worth more for this lane than a marginal accuracy gain from a black-box model.
- It directly tests the w02 claim ("why ML beats a fixed rule") — if logistic regression can't beat the w04 hand-written rule on the same data, that's an honest, useful finding, not a failure to hide.
- A Random Forest comparison is included in Section 3 as a secondary check — mainly to see whether nonlinear interactions between signals (e.g. staleness only mattering when CTR-gap is also high) add anything the linear model misses. If it doesn't meaningfully beat logistic regression, the added complexity isn't worth it (the card explicitly warns against rewarding complexity alone).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


MODEL_DATA = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_clicks) AS clicks_march,
            SUM(gsc_impressions) AS impressions_march,
            AVG(gsc_avg_position) AS avg_position_march,
            SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_march
        FROM read_parquet('{FACT_MARCH}')
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_april
        FROM read_parquet('{FACT_APRIL}')
        GROUP BY client_hash_id, content_hash_id
    ),
    content AS (
        SELECT
            client_hash_id, content_hash_id,
            content_type, main_intent, word_count, char_count,
            DATE_DIFF('day', content_updated_date, DATE '{DECISION_CUTOFF}') AS days_stale
        FROM read_parquet('{DIM_CONTENT}')
        WHERE is_published IS TRUE AND is_deleted IS FALSE
          AND content_updated_date <= DATE '{DECISION_CUTOFF}'   -- same leakage clamp as w04
    ),
    position_benchmark AS (
        SELECT ROUND(avg_position_march) AS position_bucket, AVG(ctr_march) AS expected_ctr
        FROM march GROUP BY ROUND(avg_position_march)
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.clicks_march,
        a.clicks_april,
        m.ctr_march,
        m.avg_position_march,
        GREATEST(pb.expected_ctr - m.ctr_march, 0) AS ctr_gap,
        c.content_type,
        c.main_intent,
        c.word_count,
        c.char_count,
        c.days_stale,
        CASE WHEN (a.clicks_april - m.clicks_march) / NULLIF(m.clicks_march, 0) < -0.15 THEN 1 ELSE 0 END AS declined
    FROM march m
    JOIN april a USING (client_hash_id, content_hash_id)
    JOIN content c USING (client_hash_id, content_hash_id)
    JOIN position_benchmark pb ON ROUND(m.avg_position_march) = pb.position_bucket
    WHERE m.clicks_march >= 5   -- filter tiny-click pages where % decline is just noise
""").df()

print("Rows:", len(MODEL_DATA))
print("Label balance:\n", MODEL_DATA["declined"].value_counts(normalize=True))
MODEL_DATA.head()





Rows: 3290
Label balance:
 declined
1    0.641641
0    0.358359
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,clicks_march,clicks_april,ctr_march,avg_position_march,ctr_gap,content_type,main_intent,word_count,char_count,days_stale,declined
0,client_20259bd6705d81d4,content_69b9947a3f1ca2d7,7.0,0.0,0.001855,7.308085,0.002305,keyword article,commercial,2588,16305,36,1
1,client_73cda7b4e4f265ea,content_2880691b1e1b34d4,11.0,10.0,0.004725,1.644819,0.004272,keyword article,transactional,<NA>,<NA>,34,0
2,client_73cda7b4e4f265ea,content_c723206618cc93a4,7.0,4.0,0.004348,1.995490,0.004649,keyword article,transactional,<NA>,<NA>,34,1
3,client_73cda7b4e4f265ea,content_f5e3d8ced2732741,12.0,11.0,0.006129,1.857984,0.002868,keyword article,transactional,<NA>,<NA>,34,0
4,client_73cda7b4e4f265ea,content_cd119c4f2bc0b27d,18.0,34.0,0.006283,3.592909,0.000228,keyword article,transactional,<NA>,<NA>,34,0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-grouped split** (`GroupShuffleSplit` on `client_hash_id`), not a random row-level split.

**Why:** pages belonging to the same client share a lot of unobserved structure — CMS platform, content team quality, industry seasonality, site-wide SEO health. A random split would let the model see other pages from the same client in training and "cheat" by learning client-specific patterns rather than genuinely transferable signal. This matters more here than a plain time-based split because the label itself is already a time-based construct (March→April change) — the leakage risk we need to guard against is *client* leakage, not *time* leakage, which is already handled by the label's construction. This also matches the pattern FlyRank's own starter pipeline (`03_train_model.py`) uses for its client-holdout split.

In [5]:
from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLS_NUMERIC = ["ctr_march", "avg_position_march", "ctr_gap", "word_count", "char_count", "days_stale"]
FEATURE_COLS_CATEGORICAL = ["content_type", "main_intent"]
TARGET_COL = "declined"

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(MODEL_DATA, groups=MODEL_DATA["client_hash_id"]))

train_df = MODEL_DATA.iloc[train_idx].reset_index(drop=True)
test_df  = MODEL_DATA.iloc[test_idx].reset_index(drop=True)

# Prove the split is actually client-disjoint before trusting anything downstream.
overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print("Client overlap between train/test (must be 0):", len(overlap))
print("Train rows:", len(train_df), " Test rows:", len(test_df))
print("Train label rate:", train_df[TARGET_COL].mean().round(3), " Test label rate:", test_df[TARGET_COL].mean().round(3))


Client overlap between train/test (must be 0): 0
Train rows: 3131  Test rows: 159
Train label rate: 0.638  Test label rate: 0.711


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Three things compared, same test set, same metric (Precision@K, K=top 10% of test rows by score — matches the w02 metric choice):
1. **w04 baseline rule**, recomputed on this exact test split (not the old CSV output — has to be a fair apples-to-apples comparison)
2. **Logistic Regression**
3. **Random Forest** (secondary check per Section 1 — only worth keeping if it beats Logistic Regression meaningfully)

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

print("NaN counts:\n", MODEL_DATA[FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL].isna().sum())

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), FEATURE_COLS_NUMERIC),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]), FEATURE_COLS_CATEGORICAL),
])

X_train, y_train = train_df[FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL], train_df[TARGET_COL]
X_test,  y_test  = test_df[FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL],  test_df[TARGET_COL]

logreg = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]

rf = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42))])
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

# Baseline rule recomputed on this exact test set, same weights as w04's verdicts
# (CTR-gap dominant, staleness minor — since w04 found staleness MIXED/leaning FALSE)
def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

baseline_scores = 0.85 * normalize(test_df["ctr_gap"]) + 0.15 * normalize(test_df["days_stale"])

def precision_at_k(y_true, scores, k_frac=0.10):
    k = max(1, int(len(scores) * k_frac))
    top_k_idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k_idx].mean()

results = pd.DataFrame({
    "method": ["w04 baseline rule", "Logistic Regression", "Random Forest"],
    "precision_at_10pct": [
        precision_at_k(y_test, baseline_scores),
        precision_at_k(y_test, logreg_scores),
        precision_at_k(y_test, rf_scores),
    ],
    "roc_auc": [
        roc_auc_score(y_test, baseline_scores),
        roc_auc_score(y_test, logreg_scores),
        roc_auc_score(y_test, rf_scores),
    ],
})
print(results.to_string(index=False))




NaN counts:
 ctr_march                0
avg_position_march       0
ctr_gap                  0
word_count            1495
char_count            1495
days_stale               0
content_type             0
main_intent             76
dtype: int64
             method  precision_at_10pct  roc_auc
  w04 baseline rule            0.666667 0.579261
Logistic Regression            0.800000 0.530396
      Random Forest            0.666667 0.489419


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Two things to look at: which features the Logistic Regression actually leans on (coefficients — directly readable, that was the whole argument for choosing it), and where it gets test-set rows wrong (false positives and false negatives), read a few by hand rather than just reporting a confusion-matrix count.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



# --- What the model leans on ---
feature_names = (
    FEATURE_COLS_NUMERIC +
    list(logreg.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(FEATURE_COLS_CATEGORICAL))
)
coefs = pd.Series(logreg.named_steps["clf"].coef_[0], index=feature_names).sort_values(key=abs, ascending=False)
print("Logistic Regression coefficients (sorted by magnitude):")
print(coefs.to_string())
# A positive coefficient pushes toward "declined"; negative pushes away.
# TODO: write 1-2 sentences here on whether this matches the w04 signal verdicts
# (ctr_gap should dominate; days_stale should barely register, per the MIXED/FALSE verdict).


# --- Where it's wrong ---
test_df_with_scores = test_df.copy()
test_df_with_scores["predicted_prob"] = logreg_scores
test_df_with_scores["predicted_label"] = (logreg_scores >= 0.5).astype(int)

false_positives = test_df_with_scores[
    (test_df_with_scores["predicted_label"] == 1) & (test_df_with_scores[TARGET_COL] == 0)
].sort_values("predicted_prob", ascending=False)

false_negatives = test_df_with_scores[
    (test_df_with_scores["predicted_label"] == 0) & (test_df_with_scores[TARGET_COL] == 1)
].sort_values("predicted_prob", ascending=True)

print("\nTop 5 false positives (model said decline, page was fine):")
print(false_positives[["client_hash_id", "content_hash_id", "ctr_gap", "days_stale", "predicted_prob"]].head())

print("\nTop 5 false negatives (model missed a real decline):")
print(false_negatives[["client_hash_id", "content_hash_id", "ctr_gap", "days_stale", "predicted_prob"]].head())



Logistic Regression coefficients (sorted by magnitude):
main_intent_missing            -0.578326
content_type_feedly article    -0.402831
main_intent_informational       0.302511
content_type_keyword article    0.147622
ctr_gap                         0.135126
word_count                      0.093230
main_intent_commercial          0.088612
main_intent_transactional      -0.086613
char_count                     -0.062449
avg_position_march             -0.039655
days_stale                      0.038820
main_intent_navigational        0.018609
ctr_march                       0.017279

Top 5 false positives (model said decline, page was fine):
              client_hash_id           content_hash_id   ctr_gap  days_stale  \
77   client_e547b89c05043229  content_77bed313672f16a0  0.007862          34   
53   client_e547b89c05043229  content_99d1b423e477c181  0.012346          34   
142  client_65de48885f4ef01b  content_f01ff965c8cff1cb  0.002782          34   
124  client_65de48885f4ef01b  c

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.